# pathlib

对应 `stdlib.md` 第一行：拼路径、读写文件。

笔记本在 `python_base/pathlib/qa.ipynb`。需要读写文件时，工作空间就是这个目录，题目文件落在旁边的 `q2/`、`q3/` 等子目录里。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/pathlib。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "pathlib" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "pathlib"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/pathlib')

## 1. 拼接，再拆开

用 `Path` 和 `/` 拼路径，不要用字符串加斜杠。

从 `root` 拼出 `root/src/app.py`。依次打印这四项：文件名、去掉后缀的主名、后缀（含点）、父目录。

In [2]:
from pathlib import Path

root = Path("/workspace/proj")

# 作答

file_path = root / "src" / "app.py"

print(f"文件名: {file_path.name}")
print(f"去掉后缀的主名: {file_path.stem}")
print(f"后缀: {file_path.suffix}")
print(f"父目录: {file_path.parent}")
#评阅
# 对。用 / 拼接，name、stem、suffix、parent 都取对了。

#参考答案
# path = root / "src" / "app.py"
# print(path.name, path.stem, path.suffix, path.parent)


文件名: app.py
去掉后缀的主名: app
后缀: .py
父目录: /workspace/proj/src


## 2. 写入再读出

目录已经建好。把 `text` 写进 `box/notes.txt`，再读回来并打印。

用 `Path` 的读写方法，不要自己调用 `open`。

In [3]:
from pathlib import Path

box = ROOT / "q2"
box.mkdir(parents=True, exist_ok=True)
text = "pathlib\n"

# 作答

file = box / "notes.txt"
if file.exists():
    file.unlink()
file.touch()

with file.open("+w", encoding="utf-8"):
    file.write_text(text)
    print(f"写入完成")

with file.open("r", encoding="utf-8") as f:
    print(f"读取内容: {f.read()}")
#评阅
# 读到的内容是对的，但没用题目要求的写法。
# file.open 就是在调用 open。with 打开的句柄没有被使用，真正写入的是 write_text。
# unlink 和 touch 也不需要，write_text 会自己建文件并覆盖。

#参考答案
# notes = box / "notes.txt"
# notes.write_text(text)
# print(notes.read_text())


写入完成
读取内容: pathlib



## 3. 一次建好多层

创建 `box/a/b`。父目录可以还不存在。同一段创建代码再执行一次，不能抛 `FileExistsError`。

最后打印 `b` 是不是目录。

In [4]:
from pathlib import Path

box = ROOT / "q3"

# 作答

dir = box / "a" / "b"

if dir.exists() and dir.is_dir():
    dir.rmdir()

dir.mkdir(parents=True)
print(f"创建目录完成")
#评阅
# mkdir(parents=True) 能把还不存在的父目录一起建出来。
# 同一段创建再执行一次也不报错，靠的是 exist_ok=True。
# 先 rmdir 再 mkdir 只创建了一次，也没有打印 b 是不是目录。

#参考答案
# target = box / "a" / "b"
# target.mkdir(parents=True, exist_ok=True)
# target.mkdir(parents=True, exist_ok=True)
# print(target.is_dir())


创建目录完成


## 4. 只看当前这一层

列出 `box` 的直接子项，不要进入子目录。

分成文件和目录两个列表，各自按名字排序后打印。

In [5]:
from pathlib import Path

box = ROOT / "q4"
(box / "src").mkdir(parents=True, exist_ok=True)
(box / "src" / "app.py").write_text("print(1)\n")
(box / "readme.md").write_text("# hi\n")
(box / "notes.txt").write_text("draft\n")

# 作答

sub_dirs = []
sub_files = []
for item in box.iterdir():
    if item.is_dir():
        sub_dirs.append(item.name)
    elif item.is_file():
        sub_files.append(item.name)

print(f"当前目录的子目录: {sorted(sub_dirs)}")
print(f"当前目录的文件: {sorted(sub_files)}")
#评阅
# 对。iterdir 只看这一层，文件和目录分开后按名字排序。

#参考答案
# files, dirs = [], []
# for item in box.iterdir():
#     (dirs if item.is_dir() else files).append(item.name)
# print(sorted(dirs))
# print(sorted(files))


当前目录的子目录: ['src']
当前目录的文件: ['notes.txt', 'readme.md']


## 5. 含子目录的全部 `.py`

找出 `box` 下面所有 `.py`，包含子目录。不要自己写递归。

打印相对 `box` 的路径，按字母排序。

In [6]:
from pathlib import Path

box = ROOT / "q5"
(box / "src").mkdir(parents=True, exist_ok=True)
(box / "src" / "app.py").write_text("print(1)\n")
(box / "src" / "util.py").write_text("x = 1\n")
(box / "readme.md").write_text("# hi\n")
(box / "tests").mkdir(exist_ok=True)
(box / "tests" / "test_app.py").write_text("def test():\n    pass\n")

# 作答

py_files_recur = []
for item in box.rglob("*.py"):
    py_files_recur.append(item.relative_to(box))

print(f"递归找到的py文件: {sorted(py_files_recur)}")
#评阅
# 对。rglob 找出子目录里的 .py，relative_to 印出相对路径，readme.md 被排除了。

#参考答案
# for path in sorted(p.relative_to(box) for p in box.rglob("*.py")):
#     print(path)


递归找到的py文件: [PosixPath('src/app.py'), PosixPath('src/util.py'), PosixPath('tests/test_app.py')]


## 6. 换后缀，再改名

`report.py` 已经在 `box` 里。

1. 只算出后缀换成 `.md` 之后的路径并打印，先不要动文件。
2. 再把 `report.py` 改名为 `report.md`，仍放在同一目录。
3. 打印原路径是否还存在。

In [7]:
from pathlib import Path

box = ROOT / "q6"
box.mkdir(parents=True, exist_ok=True)
src = box / "report.py"
src.write_text("draft\n")

# 作答

new_path = src.with_suffix(".md")
print(f"更改后缀后的路径: {new_path}")

if new_path.exists():
    new_path.unlink()

src.rename(new_path)
print(f"重命名完成")

print(f"原路径是否还存在: {src.exists()}")
#评阅
# 对。with_suffix 只算出新路径，rename 之后原路径不存在。
# 目标已存在时先 unlink，是为了这一格能重复运行。

#参考答案
# dest = src.with_suffix(".md")
# print(dest)
# src.rename(dest)
# print(src.exists())


更改后缀后的路径: /Users/keyficller/Documents/AEFS-Notes/codes/python_base/pathlib/q6/report.md
重命名完成
原路径是否还存在: False


## 7. 相对路径不许跳出沙箱

`sandbox` 是允许访问的根。`inside` 和 `outside` 都是相对 `sandbox` 的路径，文件可以还不存在。

把它们各自解析成绝对路径，再判断解析结果是否仍在 `sandbox` 里面。用路径对象比较，不要把路径转成字符串再比前缀。

对两个路径各打印一行：解析后的路径，以及是否还在沙箱内。

In [ ]:
from pathlib import Path

sandbox = (ROOT / "q7").resolve()
sandbox.mkdir(parents=True, exist_ok=True)
(sandbox / "ok.txt").write_text("in\n")

inside = Path("ok.txt")
outside = Path("../secret.txt")

# 作答

inside_abs = (sandbox / inside).resolve()
outside_abs = (sandbox / outside).resolve()

print(f"解析后的路径: {inside_abs}")
print(f"是否在沙箱内: {inside_abs.is_relative_to(sandbox)}")

print(f"解析后的路径: {outside_abs}")
print(f"是否在沙箱内: {outside_abs.is_relative_to(sandbox)}")
#评阅
# 对。先拼到 sandbox 上再 resolve，然后用 is_relative_to 判断。
# ok.txt 在里面，../secret.txt 解析到沙箱外面。

#参考答案
# for rel in (inside, outside):
#     full = (sandbox / rel).resolve()
#     print(full, full.is_relative_to(sandbox))
